# Pendulum reward heatmap — what IQ-Learn actually learned

Generates a 3-panel figure showing the value function and policy that IQ-Learn extracted from K=10 expert trajectories, side-by-side with the (true, unseen) Pendulum reward function.

**Goal:** visual proof that IQ-Learn recovers the goal structure of the task without ever seeing the reward.

**Steps:**
1. Mount Drive + cd to project
2. Train (or load) a Pendulum K=10 seed=42 agent — ~25 min if not cached
3. Compute soft value, policy, and true reward on a (θ, θ̇) grid
4. Plot and save as PDF for the report

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
%cd /content/drive/MyDrive/imitation_learning

In [ ]:
!pip install gymnasium -q

In [ ]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt

from iq_learn import IQLearnAgent, train_iq_learn

os.makedirs('models', exist_ok=True)
os.makedirs('plots', exist_ok=True)

CKPT_PATH = 'models/iqlearn_pendulum_K10_seed42.pt'
EXPERT_PATH = 'expert_data/Pendulum-v1_K10.npz'

print(f'Checkpoint exists: {os.path.exists(CKPT_PATH)}')
print(f'Expert data exists: {os.path.exists(EXPERT_PATH)}')

## 2. Train (if no checkpoint) — ~25 min

This runs the same Pendulum K=10 seed=42 configuration that produced our best Pendulum result (max reward ≈ −125). Saves the agent's networks for the plotting step.

If `models/iqlearn_pendulum_K10_seed42.pt` already exists, the cell skips training.

In [ ]:
if os.path.exists(CKPT_PATH):
    print('Checkpoint found — skipping training.')
else:
    print('No checkpoint found — training for ~25 minutes...')
    agent, log = train_iq_learn(
        env_name='Pendulum-v1',
        expert_npz_path=EXPERT_PATH,
        seed=42,
        total_steps=30_000,
        eval_interval=2000,
        eval_episodes=5,
        hidden_dim=256,
        batch_size=256,
        lr=1e-4,
        chi2_coef=0.5,
        alpha=0.2,
        auto_alpha=True,
        verbose=True,
    )
    torch.save({
        'actor': agent.actor.state_dict(),
        'critic': agent.critic.state_dict(),
        'critic_target': agent.critic_target.state_dict(),
        'alpha': agent.alpha,
    }, CKPT_PATH)
    print(f'\nSaved checkpoint to {CKPT_PATH}')
    print(f'Final eval reward: {log["eval_reward"][-1]:.2f}')
    print(f'Max eval reward:   {max(log["eval_reward"]):.2f}')

## 3. Load agent

In [ ]:
agent = IQLearnAgent(
    state_dim=3, action_dim=1, discrete=False,
    action_low=np.array([-2.0]), action_high=np.array([2.0]),
    hidden_dim=256, lr=1e-4, alpha=0.2, auto_alpha=True,
    chi2_coef=0.5,
)
ckpt = torch.load(CKPT_PATH, weights_only=True)
agent.actor.load_state_dict(ckpt['actor'])
agent.critic.load_state_dict(ckpt['critic'])
agent.critic_target.load_state_dict(ckpt['critic_target'])
agent.alpha = ckpt.get('alpha', 0.2)
print(f'Agent loaded. α = {agent.alpha:.3f}')

## 4. Compute the (θ, θ̇) grids

**Pendulum-v1 state space:**
- θ ∈ [−π, π]: angle (θ=0 means pendulum upright)
- θ̇ ∈ [−8, 8]: angular velocity
- observation passed to the network: `[cos θ, sin θ, θ̇]`

**Goal state**: (θ=0, θ̇=0) — pendulum upright and stationary.

In [ ]:
N = 100
thetas = np.linspace(-np.pi, np.pi, N)
theta_dots = np.linspace(-8.0, 8.0, N)
T, TD = np.meshgrid(thetas, theta_dots)

# Convert to Pendulum observations [cos θ, sin θ, θ̇]
obs = np.stack([np.cos(T), np.sin(T), TD], axis=-1).reshape(-1, 3)
obs_t = torch.FloatTensor(obs)

# ── Learned soft value V(s) = E_a~π[Q(s,a) - α log π(a|s)] ──────
with torch.no_grad():
    # average over multiple action samples for stable V estimate
    n_samples = 16
    V_samples = []
    for _ in range(n_samples):
        actions_unit, log_probs = agent.actor.sample(obs_t)
        actions_scaled = agent._scale_action(actions_unit)
        q1, q2 = agent.critic(obs_t, actions_scaled)
        q = torch.min(q1, q2)
        V_samples.append(q - agent.alpha * log_probs)
    V = torch.stack(V_samples).mean(0)
V_grid = V.squeeze().numpy().reshape(T.shape)

# ── Learned deterministic policy μ(s) (mean torque) ───────────────
with torch.no_grad():
    a_unit = agent.actor.deterministic(obs_t)
    a_scaled = agent._scale_action(a_unit)
policy_grid = a_scaled.squeeze().numpy().reshape(T.shape)

# ── True Pendulum reward r(s, a=0) for comparison ────────────────
# Gym source: r = -(θ² + 0.1·θ̇² + 0.001·a²)
true_reward_grid = -(T**2 + 0.1 * TD**2)

print('Grids computed.')
print(f'V range: [{V_grid.min():.2f}, {V_grid.max():.2f}]')
print(f'Policy range: [{policy_grid.min():.2f}, {policy_grid.max():.2f}]')

## 5. Plot — 3-panel figure for the report

In [ ]:
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 300, 'font.size': 10})

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), constrained_layout=True)

# ── Panel 1: Learned soft value ──────────────────────────
im0 = axes[0].pcolormesh(T, TD, V_grid, cmap='viridis', shading='auto')
axes[0].set_title('Soft value  $V(s)$  (IQ-Learn, K=10)', fontsize=12)
axes[0].set_xlabel(r'$\theta$  (angle, rad)')
axes[0].set_ylabel(r'$\dot\theta$  (angular velocity)')
axes[0].axhline(0, color='white', lw=0.5, alpha=0.4)
axes[0].axvline(0, color='white', lw=0.5, alpha=0.4)
axes[0].plot(0, 0, marker='*', color='white', markersize=12, markeredgecolor='black')
fig.colorbar(im0, ax=axes[0], shrink=0.85)

# ── Panel 2: Learned policy (deterministic torque) ─────────
im1 = axes[1].pcolormesh(T, TD, policy_grid, cmap='RdBu_r',
                          shading='auto', vmin=-2, vmax=2)
axes[1].set_title(r'Policy  $\mu(s)$  (mean torque)', fontsize=12)
axes[1].set_xlabel(r'$\theta$  (angle, rad)')
axes[1].set_ylabel(r'$\dot\theta$  (angular velocity)')
axes[1].axhline(0, color='k', lw=0.5, alpha=0.4)
axes[1].axvline(0, color='k', lw=0.5, alpha=0.4)
fig.colorbar(im1, ax=axes[1], shrink=0.85, label='torque (N·m)')

# ── Panel 3: True reward (for comparison) ─────────────────
im2 = axes[2].pcolormesh(T, TD, true_reward_grid, cmap='viridis', shading='auto')
axes[2].set_title(r'True reward  $r(s, a=0)$  (unseen by agent)', fontsize=12)
axes[2].set_xlabel(r'$\theta$  (angle, rad)')
axes[2].set_ylabel(r'$\dot\theta$  (angular velocity)')
axes[2].axhline(0, color='white', lw=0.5, alpha=0.4)
axes[2].axvline(0, color='white', lw=0.5, alpha=0.4)
axes[2].plot(0, 0, marker='*', color='white', markersize=12, markeredgecolor='black')
fig.colorbar(im2, ax=axes[2], shrink=0.85)

fig.suptitle('What IQ-Learn learned vs the (unseen) ground truth — Pendulum-v1',
             fontsize=13, y=1.05)

fig.savefig('plots/pendulum_reward_landscape.png')
fig.savefig('plots/pendulum_reward_landscape.pdf')
print('Saved to plots/pendulum_reward_landscape.{png,pdf}')
plt.show()

## 6. (Optional) Download the PDF for the report

In [ ]:
from google.colab import files
files.download('plots/pendulum_reward_landscape.pdf')